In [113]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [134]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [135]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
print(train_df.shape, test_df.shape)

(2000, 8) (500, 7)


In [137]:
# one row per (prompt, option) pair
def make_pairwise(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        for c in OPTION_COLS:
            rows.append({
                'text':   str(r['prompt']) + ' ' + str(r[c]),
                'qid':    r['id'],
                'option': c,
                'label':  1 if (is_train and r['answer'] == c) else 0
            })
    return pd.DataFrame(rows)

pw_train = make_pairwise(train_df, is_train=True)
pw_test  = make_pairwise(test_df,  is_train=False)
print(pw_train.shape, pw_test.shape)
print("Label dist:", pw_train['label'].value_counts().to_dict())

(10000, 4) (2500, 4)
Label dist: {0: 8000, 1: 2000}


In [138]:
tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(pw_train['text'])

X_tr  = pad_sequences(tokenizer.texts_to_sequences(pw_train['text']), maxlen=200, padding='post')
X_tst = pad_sequences(tokenizer.texts_to_sequences(pw_test['text']),  maxlen=200, padding='post')
y_tr  = pw_train['label'].values
groups = pw_train['qid'].values

print("X_tr:", X_tr.shape)

X_tr: (10000, 200)


In [140]:
class PairDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [142]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden).squeeze(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [143]:
def apk(actual, predicted, k=3):
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p == actual:
            hits += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, preds):
    return float(np.mean([apk(a, p) for a, p in zip(actuals, preds)]))

def rank_preds(pw, scores, orig_df):
    pw = pw.copy()
    pw['score'] = scores
    return [
        pw[pw['qid'] == qid].sort_values('score', ascending=False)['option'].tolist()[:3]
        for qid in orig_df['id']
    ]

In [144]:
gkf    = GroupKFold(n_splits=5)
oof    = np.zeros(len(pw_train))
tst_scores = np.zeros(len(pw_test))

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    tr_ds  = PairDataset(X_tr[ti], y_tr[ti])
    val_ds = PairDataset(X_tr[vi], y_tr[vi])

    tr_loader  = DataLoader(tr_ds,  batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = BiLSTM(20000, 128, 128).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')
    for epoch in range(10):
        model.train()
        for X_batch, y_batch in tr_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.float().to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.float().to(device)
                val_loss += criterion(model(X_batch), y_batch).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'fold{fold}.pt')

    model.load_state_dict(torch.load(f'fold{fold}.pt'))
    model.eval()

    with torch.no_grad():
        fold_scores = []
        for X_batch, _ in val_loader:
            fold_scores.extend(torch.sigmoid(model(X_batch.to(device))).cpu().numpy())
        oof[vi] = fold_scores

        test_ds = PairDataset(X_tst)
        test_loader = DataLoader(test_ds, batch_size=32)
        for X_batch in test_loader:
            tst_scores_batch = torch.sigmoid(model(X_batch.to(device))).cpu().numpy()
            tst_scores[:len(tst_scores_batch)] += tst_scores_batch / 5

    val_qids  = pw_train.iloc[vi]['qid'].unique()
    val_orig  = train_df[train_df['id'].isin(val_qids)]
    val_pw    = pw_train[pw_train['qid'].isin(val_qids)]
    val_sc    = oof[pw_train['qid'].isin(val_qids).values]
    fold_map3 = mapk(val_orig['answer'].tolist(), rank_preds(val_pw, val_sc, val_orig))
    print(f"Fold {fold+1}  MAP@3={fold_map3:.4f}")

oof_map3 = mapk(train_df['answer'].tolist(), rank_preds(pw_train, oof, train_df))
print(f"\nOOF MAP@3: {oof_map3:.4f}")

Fold 1  MAP@3=0.4108
Fold 2  MAP@3=0.4229
Fold 3  MAP@3=0.4225
Fold 4  MAP@3=0.4387
Fold 5  MAP@3=0.4483

OOF MAP@3: 0.4287


In [145]:
test_preds = rank_preds(pw_test, tst_scores, test_df)

submission = pd.DataFrame({
    'ID':         test_df['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10).to_string(index=False))

 ID Prediction
  1      D C B
  2      A C B
  3      B E C
  4      A B E
  5      D E C
  6      B E D
  7      A B C
  8      A B C
  9      A B C
 10      A B C
